# 02 Silver Layer — Cleanse & Validate

**役割：** Bronze層の生データを読み込み、品質を保証する。ビジネスロジックは持たない。  
**設計原則：** このレイヤーを通過したデータは「信頼できる」状態になっている。

---

### このノートブックの処理フロー
```
Delta Lake (Bronze)
      │ Unity Catalog Volume
      │ /Volumes/workspace/portfolio/raw_data
      │
      │ 1. 重複除去（ticker × date）
      │ 2. Null処理（adj_close の前日値補完）
      │ 3. 型の保証（date / double）
      │ 4. 対数リターン計算
      │ 5. 異常値フラグ付与
      ▼
Delta Lake (Silver) ─── Unity Catalog Table
      │ workspace.portfolio.silver_stock_returns
      │ overwrite / partition: ticker
      ▼
データ品質確認（件数・Null・異常値）
```

### 設計判断メモ
- **重複除去をSilver層で行う理由**：Bronze層はappend-onlyで生データを保持する設計のため、重複制御の責務はSilver層が持つ
- **ffill（前日値補完）の位置**：Bronze層では行わない。生データの欠損を保持しておくことで、Bronze層からの再実行時に補完ロジックを変更できる
- **対数リターンを採用する理由**：時間加法性があり、長期統計・モンテカルロ法に適している。単純リターン`(P_t - P_{t-1}) / P_{t-1}`は長期では加算できないが、対数リターン`ln(P_t / P_{t-1})`は加算で合成できる
- **異常値を除外せずフラグ管理する理由**：除外基準はビジネス要件依存のため、Silver層では判断しない。Gold層またはStreamlit側で`is_valid=True`のみを使う設計にする
- **Silver層をoverwriteにする理由**：Bronze層と異なり、Silver層は「Bronze層から常に再生成できる」ため、overwriteで最新状態を保つ運用が適切

## 0. ライブラリのインポート

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, DoubleType, BooleanType, TimestampType
)
from datetime import datetime, timezone

spark = SparkSession.builder.getOrCreate()
print(f"Spark version: {spark.version}")

## 1. 設定値（Config）

In [0]:
# ── パス設定 ───────────────────────────────────────────────────
BRONZE_PATH = "/Volumes/workspace/portfolio/raw_data"
SILVER_TABLE = "workspace.portfolio.silver_stock_returns"

# ── 異常値判定閾値 ─────────────────────────────────────────────
# 1日の対数リターンが±20%を超える場合を異常値とする
# 根拠：株式市場のサーキットブレーカーが概ね±20%前後で発動するため
ANOMALY_THRESHOLD = 0.20

PROCESSED_AT = datetime.now(timezone.utc)

print(f"Bronze path    : {BRONZE_PATH}")
print(f"Silver path    : {SILVER_TABLE}")
print(f"異常値閾値     : ±{ANOMALY_THRESHOLD * 100:.0f}%")
print(f"処理日時       : {PROCESSED_AT}")

## 2. Bronze層からデータ読み込み

In [0]:
sdf_bronze = spark.read.format("delta").load(BRONZE_PATH)

display(sdf_bronze.groupBy("ticker").count().orderBy("ticker"))
print("\nスキーマ確認:")
sdf_bronze.printSchema()

print("\n=== サンプル（先頭5件） ===")
display(sdf_bronze.orderBy("ticker", "date").limit(5))

## 3. 重複除去

Bronze層はappend-onlyのため、同一銘柄×日付のデータが複数存在する可能性がある。  
`ticker × date`の組み合わせでユニークにする。複数存在する場合は`ingested_at`が最新のものを残す。

**pandas との対応：**
```python
# pandas
df.sort_values('ingested_at').drop_duplicates(subset=['ticker', 'date'], keep='last')

# Spark（同じ意図）
Window.partitionBy('ticker', 'date').orderBy(F.desc('ingested_at'))
```

In [0]:
# ticker × date 内で ingested_at が最新のレコードを残す
w_dedup = Window.partitionBy("ticker", "date").orderBy(F.desc("ingested_at"))

sdf_dedup = (
    sdf_bronze
    .withColumn("row_num", F.row_number().over(w_dedup))
    .filter(F.col("row_num") == 1)           # 各グループの1行目（最新）だけ残す
    .drop("row_num")                          # 作業用列を削除
)

bronze_count = sdf_bronze.count()
dedup_count  = sdf_dedup.count()
display(
    sdf_bronze.groupBy("ticker").count().withColumnRenamed("count", "before")
    .join(sdf_dedup.groupBy("ticker").count().withColumnRenamed("count", "after"), "ticker")
    .withColumn("removed", F.col("before") - F.col("after"))
    .orderBy("ticker")
)

## 4. Null処理（前日値補完）

`adj_close` のNullは取引停止日・祝日などによる欠損。前日の終値で補完する（ffill）。  

**なぜBronze層でやらないか：**  
Bronze層は生データの保管庫なので、欠損もそのまま保持する。補完ロジックを変えたい場合に Bronze を再利用して Silver を再生成できる。

**pandas との対応：**
```python
# pandas
df['adj_close'] = df.groupby('ticker')['adj_close'].ffill()

# Spark（同じ意図）
Window.partitionBy('ticker').orderBy('date') + F.last(..., ignorenulls=True)
```

In [0]:
# 銘柄ごとに日付順で前日値補完（ffill）
w_ffill = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
    .rowsBetween(Window.unboundedPreceding, 0)  # 現在行より前の全行を対象
)

sdf_filled = (
    sdf_dedup
    .withColumn(
        "adj_close",
        F.last(F.col("adj_close"), ignorenulls=True).over(w_ffill)
    )
)

# 補完後のNull確認
null_count = sdf_filled.filter(F.col("adj_close").isNull()).count()
print(f"adj_close の残存Null件数: {null_count} 件")
print("（0件であれば補完完了）")

## 5. 対数リターン計算

銘柄ごとに日付順で1日前の`adj_close`との対数比を計算する。

**計算式：** `daily_return = ln(P_t / P_{t-1})`

**なぜ対数リターンか：**  
- 単純リターン `(P_t - P_{t-1}) / P_{t-1}` は期間を跨いで足し算できない  
- 対数リターンは足し算で合成できる（時間加法性）  
- モンテカルロ法でのシミュレーションに適している（正規分布を仮定しやすい）

**pandas との対応：**
```python
# pandas（既存Streamlitアプリと同じロジック）
log_returns = np.log(data / data.shift(1))

# Spark（同じ意図）
F.log(F.col('adj_close') / F.lag('adj_close', 1).over(w_return))
```

In [0]:
# 銘柄ごとに日付順で対数リターンを計算
w_return = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
)

sdf_returns = (
    sdf_filled
    .withColumn(
        "daily_return",
        F.log(F.col("adj_close") / F.lag("adj_close", 1).over(w_return))
        # F.lag()：1行前のadj_closeを参照する（pandasのshift(1)に相当）
        # 各銘柄の最初の行はlag値がNullになるため daily_return も Null になる（正常）
    )
)

# 確認：最初の行はNullになるはず
print("=== 対数リターン確認（SPYの先頭5件） ===")
(
    sdf_returns
    .filter(F.col("ticker") == "SPY")
    .select("ticker", "date", "adj_close", "daily_return")
    .orderBy("date")
    .show(5)
)

## 6. 異常値フラグ付与

対数リターンの絶対値が閾値（±20%）を超えるレコードに`is_valid=False`を付与する。  
**除外はしない。** Gold層・Streamlit側で`is_valid=True`のみを使う設計にする。

In [0]:
sdf_flagged = (
    sdf_returns
    .withColumn(
        "is_valid",
        F.when(
            F.col("daily_return").isNull(),          # 最初の行（lag値なし）は別扱い
            F.lit(None).cast(BooleanType())
        ).otherwise(
            F.abs(F.col("daily_return")) < ANOMALY_THRESHOLD
        )
    )
)

# 異常値件数確認
print("=== 異常値件数（銘柄別） ===")
(
    sdf_flagged
    .groupBy("ticker")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("is_valid") == False, 1).otherwise(0)).alias("anomaly_count")
    )
    .orderBy("ticker")
    .show()
)

## 7. Silver層スキーマを整形して書き込み準備

Silver層に必要な列だけを選択し、メタデータ列を付与する。

In [0]:
sdf_silver = (
    sdf_flagged
    .select(
        F.col("ticker"),
        F.col("date"),
        F.col("adj_close"),
        F.col("daily_return"),
        F.col("is_valid"),
    )
    .withColumn("processed_at", F.lit(PROCESSED_AT).cast(TimestampType()))
)

print("Silver層スキーマ:")
sdf_silver.printSchema()

print("\n=== サンプル（SPY 先頭10件） ===")
(
    sdf_silver
    .filter(F.col("ticker") == "SPY")
    .orderBy("date")
    .show(10)
)

## 8. Delta Lake（Silver）への書き込み

**Silver層はoverwriteにする理由：**  
Silver層は「Bronze層から常に再生成できる」派生データ。  
Bronze層と異なり、生データを保護する必要がないため、最新の処理結果で上書きする運用が適切。

In [0]:
(
    sdf_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ticker")
    .saveAsTable(SILVER_TABLE)
)
print(f"✅ Silver層への書き込み完了: {SILVER_TABLE}")

## 9. 書き込み後の品質確認

In [0]:
sdf_check = spark.read.table(SILVER_TABLE)

display(sdf_check.groupBy("ticker").agg(
    F.count("*").alias("total_rows"),
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date"),
    F.round(F.mean("daily_return") * 252, 4).alias("annual_return_approx"),
    F.sum(F.when(F.col("is_valid") == False, 1).otherwise(0)).alias("anomalies")
).orderBy("ticker"))

print("\n=== Null確認 ===")
display(sdf_check.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["adj_close", "daily_return", "is_valid"]
]))
print("※ daily_return / is_valid のNull は各銘柄の初日分（正常）")

## 10. Delta Lake メタデータ確認

In [0]:
spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

---

## ✅ Silver層 完了チェックリスト

- [ ] Bronze層から4銘柄分のデータを読み込めた
- [ ] 重複除去後の件数を確認した
- [ ] adj_close の Null が 0 件になった（ffill完了）
- [ ] daily_return が計算されている（各銘柄の初日のみNull、正常）
- [ ] 異常値件数を銘柄別に確認した
- [ ] Silver層への書き込みが完了した
- [ ] annual_return_approx が銘柄ごとに妥当な値になっている（SPYなら概ね10〜15%程度）

---

**次のステップ：** `03_gold_aggregate.ipynb` — Silver層から統計量・相関行列・ポートフォリオ指標を計算し、Streamlitが消費するGold層を構築する